# RNA-seq Feature Counting Benchmark: polars-bio vs featureCounts vs HTSeq-count

This notebook benchmarks **polars-bio** gene-level feature counting against two established reference tools:
**featureCounts** (subread 2.1.1) and **HTSeq-count** (2.1.2).

The goal is to prove that polars-bio produces **correct gene-level counts** with **competitive speed**
and **lower memory** than featureCounts.

## Dataset

| Property | Value |
|----------|-------|
| Experiment | ENCODE ENCSR329MHM |
| Cell line | HepG2 |
| Library | Single-end, 76 bp |
| Reads | ~36M |
| Genome | GRCh38 |
| BAM accession | ENCFF588KDY (Rep1) |
| Annotation | GENCODE v49 comprehensive GTF (GRCh38.p14) |

## 1. Configuration

All data paths are configured via the `POLARS_BIO_BENCHMARK_DATA` environment variable.
Set this variable before running the notebook, or use the default path.

In [ ]:
import os
DATA_DIR = os.environ.get("POLARS_BIO_BENCHMARK_DATA", "/Users/mwiewior/research/data/polars-bio/rnaseq")
BAM_FILE = os.path.join(DATA_DIR, "ENCFF588KDY.bam")
BAM_INDEX = os.path.join(DATA_DIR, "ENCFF588KDY.bam.bai")
GTF_FILE = os.path.join(DATA_DIR, "gencode.v49.annotation.gtf")
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")

## 2. Download BAM (DATA-01)

Download the ENCODE accession **ENCFF588KDY** -- Rep1 BAM from experiment ENCSR329MHM.
This is a coordinate-sorted BAM (~3.2 GB) aligned to GRCh38 by the ENCODE STAR pipeline.
Since the ENCODE pipeline uses `--outSAMtype BAM SortedByCoordinate`, the BAM is already
coordinate-sorted and we only need to index it (no `samtools sort` required).

In [ ]:
%%bash -s "$DATA_DIR"
cd "$1"
if [ ! -f ENCFF588KDY.bam ]; then
    wget -O ENCFF588KDY.bam "https://www.encodeproject.org/files/ENCFF588KDY/@@download/ENCFF588KDY.bam"
fi

### Verify BAM integrity (optional MD5 check)

Expected MD5: `90779101e28e5b425be8e51310314b89` (from ENCODE portal).

In [ ]:
%%bash -s "$DATA_DIR"
cd "$1"
# Expected MD5: 90779101e28e5b425be8e51310314b89
md5 ENCFF588KDY.bam 2>/dev/null || md5sum ENCFF588KDY.bam

### Verify BAM is coordinate-sorted

ENCODE STAR pipeline outputs sorted BAMs, so we skip `samtools sort`.
The header should contain `SO:coordinate`.

In [ ]:
%%bash -s "$DATA_DIR"
cd "$1"
samtools view -H ENCFF588KDY.bam | head -1
# Expected: @HD	VN:1.4	SO:coordinate

### Index BAM

Creates the `.bai` index file required for random access and region queries.

In [ ]:
%%bash -s "$DATA_DIR"
cd "$1"
if [ ! -f ENCFF588KDY.bam.bai ]; then
    samtools index ENCFF588KDY.bam
fi

### Validate read count

Expect ~36M total reads from this ENCODE replicate.

In [ ]:
%%bash -s "$DATA_DIR"
cd "$1"
samtools flagstat ENCFF588KDY.bam

## 3. Download GTF (DATA-02)

Download **GENCODE v49** comprehensive gene annotation (CHR variant -- reference chromosomes only).
This is the GRCh38.p14 annotation with chr-prefixed contigs matching the ENCODE BAM.

In [ ]:
%%bash -s "$DATA_DIR"
cd "$1"
if [ ! -f gencode.v49.annotation.gtf ]; then
    wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.annotation.gtf.gz
    gunzip gencode.v49.annotation.gtf.gz
fi

### Verify GTF readability with polars-bio

Scan the GTF with `polars-bio` to confirm it can be read and parsed correctly.
We extract exon rows and count unique gene IDs as a sanity check.

In [ ]:
import polars_bio as pb
gtf = pb.scan_gtf(GTF_FILE, attr_fields=["gene_id"])
exons = gtf.filter(pb.col("feature") == "exon").collect()
print(f"Total exon rows: {len(exons)}")
print(f"Unique gene_ids: {exons['gene_id'].n_unique()}")
print(f"Sample rows:\n{exons.head(3)}")

## 4. Chromosome Naming Validation (DATA-03)

Both the ENCODE GRCh38 BAM and the GENCODE v49 CHR GTF should use **chr-prefixed** contigs.
A mismatch would silently produce **zero counts** in all downstream tools -- this is a critical
validation step.

In [ ]:
import polars_bio as pb
import subprocess, re

# BAM contigs from header
result = subprocess.run(
    ["samtools", "view", "-H", BAM_FILE],
    capture_output=True, text=True, check=True
)
bam_contigs = sorted(set(
    re.findall(r"@SQ\tSN:(\S+)", result.stdout)
))
print(f"BAM contigs ({len(bam_contigs)}): {bam_contigs[:5]} ...")

# GTF seqnames
gtf_contigs = sorted(
    pb.scan_gtf(GTF_FILE)
    .select("seqname")
    .unique()
    .collect()["seqname"]
    .to_list()
)
print(f"GTF seqnames ({len(gtf_contigs)}): {gtf_contigs[:5]} ...")

# Check overlap
common = set(bam_contigs) & set(gtf_contigs)
print(f"\nCommon contigs: {len(common)}")
assert len(common) > 0, "FATAL: No common chromosome names between BAM and GTF!"
assert any(c.startswith("chr") for c in common), "Expected chr-prefixed contigs"
print("PASS: Chromosome naming is consistent (chr-prefixed)")

## 5. Reference Tool Installation

Install the reference counting tools used for benchmark comparison.
These commands are provided as documentation -- run them manually in your terminal
before proceeding to the benchmark sections.

### featureCounts 2.1.1 (via conda/mamba)

```bash
conda install -c bioconda subread=2.1.1
# OR
mamba install -c bioconda subread=2.1.1
```

### HTSeq-count 2.1.2 (via uv or pip)

```bash
uv pip install HTSeq==2.1.2
# OR
pip install HTSeq==2.1.2
```

### System tools (via brew or apt)

```bash
# macOS
brew install hyperfine
brew install coreutils  # provides gtime

# Linux
apt install hyperfine time
```

### Verify tool versions

In [ ]:
%%bash
echo "featureCounts:"
featureCounts -v 2>&1 | head -1
echo ""
echo "HTSeq-count:"
htseq-count --version
echo ""
echo "samtools:"
samtools --version | head -1

## 6. Dataset Summary

Verify that all required dataset files are in place before proceeding to benchmark phases.

| File | Description | Expected Size |
|------|-------------|---------------|
| `ENCFF588KDY.bam` | ENCODE Rep1 BAM (GRCh38, SE76) | ~3.2 GB |
| `ENCFF588KDY.bam.bai` | BAM index | ~6 MB |
| `gencode.v49.annotation.gtf` | GENCODE v49 comprehensive GTF | ~1.6 GB |

In [ ]:
import os
files = {
    "BAM": BAM_FILE,
    "BAM index": BAM_INDEX,
    "GTF": GTF_FILE,
}
print("Dataset files:")
for name, path in files.items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) / (1024**3) if exists else 0
    print(f"  {name}: {'OK' if exists else 'MISSING'} ({size:.2f} GB) - {path}")